# 02_ner_training.ipynb

## Setup

In [ ]:
import os
import sys
from pathlib import Path

# Mount colab
if True:
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')

PROJECT_ROOT: Path = Path("/content/drive/MyDrive/python/projects/nlp_projects/med/biomedical-extraction-pipeline")
os.chdir(PROJECT_ROOT)

# Add the PROJECT_ROOT directory to the Python path to import modules
if str(PROJECT_ROOT) in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"ℹ️ Current Python version: {sys.version}")

In [ ]:
# Install reqs (NOTE: restart after this)
if False:
    %pip install -r {"requirements/02_ner_training.txt"}

In [ ]:
# --- src ---
from src.config import load_configs
from src.utils import (
    dynamic_module_reloader, 
    seed_everything, 
    plot_length_distribution_with_percentiles,
    estimate_optimization_steps
    )
from src.modelling.named_entity_recognition import tokenize_and_align_labels
from src.modelling.eval import compute_metrics_ner

# --- other ---
import time
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt
import torch
from functools import partial
from dotenv import load_dotenv
from pprint import pprint
from datasets import DatasetDict
from transformers import (
    AutoConfig, 
    AutoTokenizer,
    AutoModelForTokenClassification, 
    DataCollatorForTokenClassification, 
    EarlyStoppingCallback,
    TrainingArguments, 
    Trainer
    )
from peft import get_peft_model, LoraConfig, TaskType

MODULES_TO_RELOAD = [
    "src",
    "src.config",
    "src.utils",
    "src.model_utils",
    "src.model_utils.eval"
    "src.model_utils.named_entity_recognition",
]

dynamic_module_reloader(modules=MODULES_TO_RELOAD, verbose=1)
config = load_configs()
load_dotenv()
seed_everything(config.env.seed)
pprint(config.to_dict())

# Constants
DEVICE: str = 'cuda' if torch.cuda.is_available() else 'cpu'
SMOKE_TEST: bool = False
SAVE_LOCAL: bool = True
PUSH_TO_HUB: bool = True

if SMOKE_TEST:
    current_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    config.model.ner_model_name += f"-smoke-test-{current_time}"
    config.tracking.project = "smoke-tests"

# Assertions to prevent incompatible settings
assert not (SMOKE_TEST and PUSH_TO_HUB), "Cannot run in SMOKE_TEST mode and PUSH_TO_HUB at the same time."
assert not (SMOKE_TEST and SAVE_LOCAL), "Cannot run in SMOKE_TEST mode and SAVE_LOCAL at the same time."
assert not (DEVICE=='cpu' and not SMOKE_TEST), "Running on CPU without SMOKE_TEST mode can be very slow. Please set SMOKE_TEST to True or use a GPU."

print("\n\t--- Notebook settings ---")
print(f"Using device: {DEVICE}")
print(f"SMOKE_TEST mode: {SMOKE_TEST}")

In [ ]:
!nvidia-smi

## Data

In [ ]:
# Load the processed dataset (bigbio/BC5CDR)
flattened_dataset = DatasetDict.load_from_disk(config.data.flattened_dataset_path)
flattened_dataset

In [ ]:
# id2label & label2id
id2label = {i: label for i, label in enumerate(config.model.ner_label_names)}
label2id = {label: i for i, label in enumerate(config.model.ner_label_names)}

print(f"id2label: {id2label}")
print(f"label2id: {label2id}")

In [ ]:
# Instantiate the tokenizer and data_collator [OPTIONAL]
model_checkpoint = config.model.ner_model_checkpoint
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
data_collator = DataCollatorForTokenClassification(tokenizer)

# Tokenize (+ offsets and BIO tagging)
tokenize_fn = partial(
    tokenize_and_align_labels, 
    tokenizer=tokenizer,
    padding=data_collator is None,
    truncate=True
    )

tokenized_dataset = flattened_dataset.map(tokenize_fn, batched=True)

# Add ner-tags (str) [OPTIONAL]
if True:
    def int_to_string(example):
        example["ner-tags"] = [
            "IGNORE" if label == -100 else id2label[label] 
            for label in example["labels"]
        ]
        return example
    tokenized_dataset = tokenized_dataset.map(int_to_string)

tokenized_dataset

With `lambda` it would be like:

```python
tokenized_dataset = flattened_dataset['train'].map(
    lambda examples: tokenize_and_align_labels(examples, tokenizer=tokenizer),
    batched=True
)
```

Either [`partial`](https://stackoverflow.com/questions/15331726/how-does-functools-partial-do-what-it-does) or `lambda` work here. `partial`, however, is generally faster and more memory-efficient than a `lambda` for batched operations because it avoids the overhead of creating a new function frame every time it is called.

In [ ]:
# Sanity check of offsets and BIO tags in tokenized dataset (horizontal)
if True:
    sample = tokenized_dataset["train"][0]
    tokens = tokenizer.convert_ids_to_tokens(sample["input_ids"])
    print("Original text:", sample["text"])
    print("Tokens:", tokens)
    print("Offsets:", sample["offset_mapping"])
    print("BIO tags:", sample["labels"])
    print("BIO tags (str):", sample["ner-tags"])

    print("\nToken-BIO Tag pairs:")
    display(
        pd.DataFrame(
            zip(tokens, sample["labels"], sample["ner-tags"]),
            columns=["Token", "BIO Tag", "BIO Tag (str)"]
        ))

In [ ]:
# Check token length distribution and model max length
if True:
    print("Checking token length distribution and model max length (with truncation=True)")
    token_lengths = [len(sample["input_ids"]) for sample in tokenized_dataset["train"]]
    model_max_length = AutoConfig.from_pretrained(model_checkpoint).max_position_embeddings
    plot_length_distribution_with_percentiles(token_lengths, model_max_length=model_max_length)

In [ ]:
if SMOKE_TEST:
    n_samples = 25
    print(f"ℹ️Dataset downsampled for smoke-test ({n_samples} samples per split)")
    tokenized_dataset = DatasetDict({
        split: tokenized_dataset[split].select(range(n_samples)) for split in tokenized_dataset.keys()
    })

## Training

### LoRA configs

In [ ]:
model_config = AutoConfig.from_pretrained(
    model_checkpoint,
    num_labels=len(id2label),
    id2label=id2label,
    label2id=label2id
)

model = AutoModelForTokenClassification.from_pretrained(
    model_checkpoint, 
    config=model_config,
    device_map="auto",
) #.to(DEVICE)

peft_config = LoraConfig(
    task_type=TaskType.TOKEN_CLS, 
    r=16, 
    lora_alpha=32, 
    lora_dropout=0.1,
    target_modules=["query", "value"]
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()
# Note: You should see that only ~1-2% of total parameters are trainable!

### Training configs

In [ ]:
early_stopping = EarlyStoppingCallback(
    early_stopping_patience=2,
    early_stopping_threshold=0.001
)

compute_metrics_fn = partial(compute_metrics_ner, label_list=list(id2label.values()))

training_args = TrainingArguments(
    output_dir=f"results/ner/{config.model.ner_model_name}",
    # log_level="info",
    
    num_train_epochs=20 if not SMOKE_TEST else 3,
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    eval_accumulation_steps=4,
    weight_decay=0.01,
    warmup_steps=20,
    lr_scheduler_type='linear',
    seed=config.env.seed,
    learning_rate = 2e-4, # LoRA typically prefers slightly higher learning rates,

    # eval_steps = 10,
    eval_strategy ="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    save_total_limit=1,

    report_to = config.tracking.tracker,
    run_name = f"{config.tracking.project}/{config.model.ner_model_name}",
    project = config.tracking.project,
    trackio_space_id  = config.tracking.trackio_space_id,
    push_to_hub = False, # False: push only the best model (after training completion)
    hub_private_repo = False,
    hub_model_id = config.model.ner_model_name,
)

In [ ]:
estimate_optimization_steps(
    tokenized_dataset["train"], 
    batch_size=training_args.per_device_train_batch_size,
    gradient_accumulation_steps=training_args.gradient_accumulation_steps
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics_fn,
    data_collator=data_collator,
    callbacks=[early_stopping]
)

if True:
    start = time.time()
    trainer.train(resume_from_checkpoint=False)
    end = time.time()
    print("Training finished. Checkpoint saved to:", trainer.state.best_model_checkpoint if trainer.state else "N/A")
    print(f"Training time: {(end - start)/60:.2f} min")

    if PUSH_TO_HUB:
        trainer.push_to_hub(commit_message="Upload adapter weights and tokenizer")
        trainer.model.config.push_to_hub(
            commit_message="Pushing configs (with label mappings)",
            repo_id=trainer.args.hub_model_id
        )

In [ ]:
# Convert the history list to a DataFrame
history_df = pd.DataFrame(trainer.state.log_history)
grouped_history = history_df.groupby("epoch").first().reset_index()

# Plot train history and save in evals/images with name config.model.ner_model_name+"_"+"training_history.png" in 300 dpi
plt.figure(figsize=(10, 6))
plt.plot(grouped_history["step"], grouped_history["loss"], label="Train Loss")
plt.plot(grouped_history["step"], grouped_history["eval_loss"], label="Validation Loss")
plt.xlabel("Step")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.grid()

if True:
    plt.savefig(f"evals/images/{config.model.ner_model_name}_train_history.png", dpi=300, bbox_inches='tight')

plt.show()

## Requirements (full snapshot)

In [ ]:
if True:
    !pip freeze > snapshots/02_ner_training.txt